# Spatial filtering

Study kernel correlation, smoothing, median filtering, signed derivatives, and Laplacian sharpening.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/image_processing").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from image_processing import load_sample, show_images

try:
    import cv2 as cv
except ImportError:
    cv = None
    print("OpenCV is unavailable. NumPy examples still run; install requirements.txt for comparisons.")

## Correlation and convolution

Correlation slides the kernel without flipping it. Convolution flips the kernel first. OpenCV filter2D computes correlation.

In [ ]:
from image_processing import correlate
image = load_sample()
kernel = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
edges = correlate(image, kernel)
show_images([image, edges], ["Original", "Signed correlation"])
if cv is not None:
    builtin = cv.filter2D(image, cv.CV_64F, kernel, borderType=cv.BORDER_CONSTANT)
    np.testing.assert_allclose(edges, builtin)

## Box smoothing

Average nearby intensities. Both comparisons use zero padding.

In [ ]:
kernel = np.ones((3, 3)) / 9
smooth = correlate(image, kernel)
show_images([image, smooth], ["Original", "Box smoothing"])
if cv is not None:
    builtin = cv.filter2D(image, cv.CV_64F, kernel, borderType=cv.BORDER_CONSTANT)
    np.testing.assert_allclose(smooth, builtin, atol=1e-8)

## Median filtering

The median reduces isolated extreme pixels. Replicate edge pixels to match OpenCV medianBlur.

In [ ]:
rng = np.random.default_rng(42)
noisy = image.copy()
noisy[rng.random(image.shape) < .04] = 255
padded = np.pad(noisy, 1, mode="edge")
median = np.zeros_like(noisy)
for row in range(noisy.shape[0]):
    for col in range(noisy.shape[1]):
        median[row, col] = np.median(padded[row:row + 3, col:col + 3])
show_images([noisy, median], ["Noise", "Manual median"])
if cv is not None:
    np.testing.assert_array_equal(median, cv.medianBlur(noisy, 3))

## First and second derivatives

Keep negative responses in floating point. Casting derivatives directly to uint8 loses their sign.

In [ ]:
horizontal = correlate(image, np.array([[0, 0, 0], [-1, 0, 1], [0, 0, 0]]))
vertical = correlate(image, np.array([[0, -1, 0], [0, 0, 0], [0, 1, 0]]))
laplacian = correlate(image, np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]]))
sharpened = np.clip(image.astype(float) - laplacian, 0, 255).astype(np.uint8)
show_images([horizontal, vertical, laplacian, sharpened], ["Horizontal", "Vertical", "Laplacian", "Sharpened"])